# 01. Generate synthetic multi-sector monthly default-count data

This notebook generates `SAMPLE.csv` with the same column structure as `SP_monthly_sector_and_all.csv`.

The synthetic data are **not** intended to reproduce the empirical S&P data or the numerical results in the paper. They are provided only for code execution and workflow demonstration.


In [1]:
# ============================================================
# 0. Setup
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

SEED = 12345
rng = np.random.default_rng(SEED)

DATA_DIR = Path("./data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = DATA_DIR / "SAMPLE.csv"

# Monthly period matching the empirical workflow: 1981-01 to 2021-09, 489 months.
DATES = pd.date_range("1981-01-31", "2021-09-30", freq="M")
T = len(DATES)

print("T =", T)
print("Output:", OUT_CSV)


T = 489
Output: data\SAMPLE.csv


In [2]:
# ============================================================
# 1. Sector definitions
# ============================================================

# Same sector labels and ordering as SP_monthly_sector_and_all.csv.
SECTORS = [
    "Consumer",
    "Energy",
    "FI",
    "Forest",
    "Health",
    "High Tech",
    "Insurance",
    "Leisure",
    "Metal",
    "Real Estate",
    "Telecom",
    "Transport",
    "Utility",
]

# Approximate sector exposure weights. These are deliberately coarse and are not
# fitted to the proprietary data. They only make the synthetic file numerically plausible.
sector_weights = pd.Series({
    "Consumer": 0.115,
    "Energy": 0.065,
    "FI": 0.175,
    "Forest": 0.038,
    "Health": 0.064,
    "High Tech": 0.042,
    "Insurance": 0.128,
    "Leisure": 0.063,
    "Metal": 0.102,
    "Real Estate": 0.035,
    "Telecom": 0.039,
    "Transport": 0.040,
    "Utility": 0.094,
}, dtype=float)
sector_weights = sector_weights / sector_weights.sum()

# Baseline monthly default rates by sector, on the order of 10^{-4}--10^{-3}.
# These are synthetic design values, not empirical estimates.
base_monthly_pd = pd.Series({
    "Consumer": 0.0018,
    "Energy": 0.0021,
    "FI": 0.00055,
    "Forest": 0.0018,
    "Health": 0.0010,
    "High Tech": 0.0010,
    "Insurance": 0.00035,
    "Leisure": 0.0024,
    "Metal": 0.0016,
    "Real Estate": 0.00075,
    "Telecom": 0.0017,
    "Transport": 0.0016,
    "Utility": 0.00035,
}, dtype=float)

# Loadings used only to create plausible sector co-movement and sector rotation.
# The first factor is market-wide. The second factor rotates stress across sectors.
sector_loading_1 = pd.Series(1.0, index=SECTORS, dtype=float)
sector_loading_2 = pd.Series({
    "Consumer": 0.35,
    "Energy": 0.65,
    "FI": -0.35,
    "Forest": 0.40,
    "Health": -0.15,
    "High Tech": 0.25,
    "Insurance": -0.45,
    "Leisure": 0.55,
    "Metal": 0.60,
    "Real Estate": -0.25,
    "Telecom": 0.20,
    "Transport": 0.30,
    "Utility": -0.55,
}, dtype=float)

pd.DataFrame({
    "weight": sector_weights,
    "base_monthly_pd": base_monthly_pd,
    "loading_1": sector_loading_1,
    "loading_2": sector_loading_2,
})


,weight,base_monthly_pd,loading_1,loading_2
Consumer,0.115,0.00180,1.0,0.35
Energy,0.065,0.00210,1.0,0.65
FI,0.175,0.00055,1.0,-0.35
Forest,0.038,0.00180,1.0,0.40
Health,0.064,0.00100,1.0,-0.15
High Tech,0.042,0.00100,1.0,0.25
Insurance,0.128,0.00035,1.0,-0.45
Leisure,0.063,0.00240,1.0,0.55
Metal,0.102,0.00160,1.0,0.60
Real Estate,0.035,0.00075,1.0,-0.25


In [3]:
# ============================================================
# 2. Generate common monthly latent states
# ============================================================

def generate_ar1(T, phi, sigma, rng):
    """Stationary AR(1) with mean zero."""
    x = np.zeros(T, dtype=float)
    x[0] = rng.normal(0.0, sigma / np.sqrt(1.0 - phi**2))
    for t in range(1, T):
        x[t] = phi * x[t - 1] + rng.normal(0.0, sigma)
    return x

# Market-wide persistent credit state.
F1 = generate_ar1(T, phi=0.965, sigma=0.18, rng=rng)

# Sector-rotation persistent state.
F2 = generate_ar1(T, phi=0.935, sigma=0.20, rng=rng)

# Mild crisis pulses to make annual clustering visible in workflow diagnostics.
# The dates are synthetic and should not be interpreted as empirical event labels.
t = np.arange(T)
crisis = (
    0.75 * np.exp(-0.5 * ((t - 115) / 10.0) ** 2)
    + 1.05 * np.exp(-0.5 * ((t - 245) / 16.0) ** 2)
    + 0.85 * np.exp(-0.5 * ((t - 338) / 12.0) ** 2)
    + 0.70 * np.exp(-0.5 * ((t - 472) / 8.0) ** 2)
)

latent = pd.DataFrame({
    "date": DATES,
    "F1_market": F1,
    "F2_rotation": F2,
    "crisis_pulse": crisis,
})
latent.head()


,date,F1_market,F2_rotation,crisis_pulse
0,1981-01-31,-0.977268,0.711564,1.436609e-29
1,1981-02-28,-0.715593,0.470360,4.514460e-29
2,1981-03-31,-0.847266,0.422839,1.404527e-28
3,1981-04-30,-0.864263,0.231015,4.326247e-28
4,1981-05-31,-0.847576,0.027525,1.319319e-27


In [4]:
# ============================================================
# 3. Generate sector-level monthly observations
# ============================================================

def make_total_bonds(T, rng):
    """Create a smooth synthetic portfolio-size path on the same rough scale."""
    trend = np.linspace(1350.0, 6100.0, T)
    cycle = 280.0 * np.sin(np.linspace(0, 5.5 * np.pi, T))
    noise = generate_ar1(T, phi=0.90, sigma=65.0, rng=rng)
    total = trend + cycle + noise
    total = np.clip(total, 1200.0, 6500.0)
    return np.rint(total).astype(int)

all_bonds_target = make_total_bonds(T, rng)

# Persistent sector exposure perturbations. They keep sector shares from being exactly constant.
sector_share_noise = {
    s: generate_ar1(T, phi=0.97, sigma=0.025, rng=rng) for s in SECTORS
}

rows = []

for i, date in enumerate(DATES):
    raw_w = np.array([
        sector_weights[s] * np.exp(sector_share_noise[s][i]) for s in SECTORS
    ], dtype=float)
    raw_w = raw_w / raw_w.sum()

    # Multinomial allocation ensures that sector bonds sum exactly to the ALL total.
    sector_bonds = rng.multinomial(int(all_bonds_target[i]), raw_w)

    for s, bonds in zip(SECTORS, sector_bonds):
        bonds = int(max(bonds, 1))

        # Synthetic monthly default probability.
        log_pd = (
            np.log(base_monthly_pd[s])
            + 0.95 * sector_loading_1[s] * F1[i]
            + 0.45 * sector_loading_2[s] * F2[i]
            + 0.45 * crisis[i]
            - 0.08
            + rng.normal(0.0, 0.10)
        )
        p_default = float(np.clip(np.exp(log_pd), 1e-6, 0.075))

        # Transition probabilities. These are intentionally simple and only for workflow use.
        p_upgrade = float(np.clip(0.0035 + 0.0010 * rng.normal(), 0.0005, 0.0120))
        p_downgrade = float(np.clip(0.0060 + 0.0100 * p_default / 0.002 + 0.0020 * max(F1[i], 0), 0.0010, 0.0550))
        p_non_rated = float(np.clip(0.0050 + 0.0015 * rng.normal(), 0.0005, 0.0200))

        probs = np.array([p_default, p_upgrade, p_downgrade, p_non_rated], dtype=float)
        if probs.sum() > 0.20:
            probs = probs * (0.20 / probs.sum())
        probs = np.append(probs, 1.0 - probs.sum())

        defaulted, upgrade, downgrade, non_rated, same = rng.multinomial(bonds, probs)

        rows.append({
            "date": date.strftime("%Y-%m-%d"),
            "year": int(date.year),
            "month": int(date.month),
            "quarter": int((date.month - 1) // 3 + 1),
            "half": int(1 if date.month <= 6 else 2),
            "sector": s,
            "bonds": int(bonds),
            "defaulted": int(defaulted),
            "upgrade": int(upgrade),
            "same": int(same),
            "downgrade": int(downgrade),
            "non_rated": int(non_rated),
            "default_rate": float(defaulted / bonds),
        })

sector_df = pd.DataFrame(rows)
sector_df.head()


,date,year,month,quarter,half,sector,bonds,defaulted,upgrade,same,downgrade,non_rated,default_rate
0,1981-01-31,1981,1,1,1,Consumer,194,0,1,192,1,0,0.0
1,1981-01-31,1981,1,1,1,Energy,95,0,0,94,1,0,0.0
2,1981-01-31,1981,1,1,1,FI,317,0,2,313,2,0,0.0
3,1981-01-31,1981,1,1,1,Forest,45,0,0,44,1,0,0.0
4,1981-01-31,1981,1,1,1,Health,70,0,0,70,0,0,0.0


In [5]:
# ============================================================
# 4. Add ALL rows by summing the 13 sectors
# ============================================================

COUNT_COLS = ["bonds", "defaulted", "upgrade", "same", "downgrade", "non_rated"]
BASE_COLS = ["date", "year", "month", "quarter", "half"]
COL_ORDER = BASE_COLS + ["sector"] + COUNT_COLS + ["default_rate"]

all_df = (
    sector_df
    .groupby(BASE_COLS, as_index=False)[COUNT_COLS]
    .sum()
)
all_df["sector"] = "ALL"
all_df["default_rate"] = all_df["defaulted"] / all_df["bonds"]
all_df = all_df[COL_ORDER]

sample_df = pd.concat([all_df, sector_df[COL_ORDER]], ignore_index=True)

sector_order = {"ALL": 0, **{s: i + 1 for i, s in enumerate(SECTORS)}}
sample_df["_sector_order"] = sample_df["sector"].map(sector_order)
sample_df = (
    sample_df
    .sort_values(["date", "_sector_order"])
    .drop(columns="_sector_order")
    .reset_index(drop=True)
)

sample_df.head(20)


,date,year,month,quarter,half,sector,bonds,defaulted,upgrade,same,downgrade,non_rated,default_rate
0,1981-01-31,1981,1,1,1,ALL,1482,0,7,1459,13,3,0.000000
1,1981-01-31,1981,1,1,1,Consumer,194,0,1,192,1,0,0.000000
2,1981-01-31,1981,1,1,1,Energy,95,0,0,94,1,0,0.000000
3,1981-01-31,1981,1,1,1,FI,317,0,2,313,2,0,0.000000
4,1981-01-31,1981,1,1,1,Forest,45,0,0,44,1,0,0.000000
5,1981-01-31,1981,1,1,1,Health,70,0,0,70,0,0,0.000000
6,1981-01-31,1981,1,1,1,High Tech,52,0,0,52,0,0,0.000000
7,1981-01-31,1981,1,1,1,Insurance,169,0,0,166,1,2,0.000000
8,1981-01-31,1981,1,1,1,Leisure,86,0,2,81,2,1,0.000000
9,1981-01-31,1981,1,1,1,Metal,158,0,2,156,0,0,0.000000


In [6]:
# ============================================================
# 5. Validation checks
# ============================================================

assert list(sample_df.columns) == COL_ORDER
assert sample_df.shape[0] == T * (len(SECTORS) + 1)
assert sample_df["date"].nunique() == T
assert sample_df.groupby("date")["sector"].nunique().eq(len(SECTORS) + 1).all()

# Counts must be internally consistent.
check_counts = sample_df["defaulted"] + sample_df["upgrade"] + sample_df["same"] + sample_df["downgrade"] + sample_df["non_rated"]
assert np.all(check_counts.to_numpy() == sample_df["bonds"].to_numpy())

# ALL must equal the sum of sector rows for all count columns.
sector_sum = sample_df[sample_df["sector"] != "ALL"].groupby("date")[COUNT_COLS].sum()
all_sum = sample_df[sample_df["sector"] == "ALL"].set_index("date")[COUNT_COLS]
assert (sector_sum == all_sum).all().all()

# Default rate must equal defaulted / bonds.
assert np.allclose(sample_df["default_rate"], sample_df["defaulted"] / sample_df["bonds"])

all_monthly = sample_df[sample_df["sector"] == "ALL"].copy()
annual = (
    all_monthly
    .assign(year=pd.to_datetime(all_monthly["date"]).dt.year)
    .groupby("year")
    .agg(defaulted=("defaulted", "sum"), bonds_start=("bonds", "first"))
)
annual["annual_default_rate"] = annual["defaulted"] / annual["bonds_start"]

summary_by_sector = (
    sample_df
    .groupby("sector")
    .agg(
        bonds_mean=("bonds", "mean"),
        bonds_min=("bonds", "min"),
        bonds_max=("bonds", "max"),
        defaulted_mean=("defaulted", "mean"),
        defaulted_sum=("defaulted", "sum"),
        default_rate_mean=("default_rate", "mean"),
        default_rate_max=("default_rate", "max"),
    )
    .sort_values("bonds_mean", ascending=False)
)

print("sample_df shape:", sample_df.shape)
print("monthly ALL mean default rate:", all_monthly["default_rate"].mean())
print("annual ALL mean default rate:", annual["annual_default_rate"].mean())
print("annual ALL default-rate std:", annual["annual_default_rate"].std())
summary_by_sector


sample_df shape: (6846, 13)
monthly ALL mean default rate: 0.0013055237209870826
annual ALL mean default rate: 0.015757046397174887
annual ALL default-rate std: 0.009198449752204149


,bonds_mean,bonds_min,bonds_max,defaulted_mean,defaulted_sum,default_rate_mean,default_rate_max
sector,,,,,,,
ALL,3806.040900,1200,5967,5.059305,2474,0.001306,0.007032
FI,655.372188,248,1171,0.351738,172,0.000513,0.005510
Insurance,478.584867,116,815,0.149284,73,0.000310,0.007380
Consumer,427.190184,135,718,0.895706,438,0.002071,0.013423
Metal,405.486708,124,801,0.730061,357,0.001792,0.013263
Utility,346.611452,94,692,0.143149,70,0.000409,0.006349
Energy,261.527607,63,489,0.654397,320,0.002444,0.028708
Health,247.449898,64,399,0.267894,131,0.001088,0.015625
Leisure,245.869121,53,442,0.701431,343,0.002813,0.022059


In [7]:
# ============================================================
# 6. Save SAMPLE.csv
# ============================================================

sample_df.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}")

# If you want the file in the current Colab directory instead of ./data, uncomment:
# sample_df.to_csv("SAMPLE.csv", index=False)


Saved: data\SAMPLE.csv


## Loader convention for later notebooks

Later notebooks should use a single input path variable. If the proprietary file exists, set `INPUT_CSV` to that file. Otherwise, use `data/SAMPLE.csv`.


In [8]:
# ============================================================
# 7. Recommended data-loading cell for 02 and later notebooks
# ============================================================

from pathlib import Path
import pandas as pd

REAL_CSV = Path("./data/SP_monthly_sector_and_all.csv")
SAMPLE_CSV = Path("./data/SAMPLE.csv")

INPUT_CSV = REAL_CSV if REAL_CSV.exists() else SAMPLE_CSV
print("Using:", INPUT_CSV)

df = pd.read_csv(INPUT_CSV, parse_dates=["date"])

required_cols = [
    "date", "year", "month", "quarter", "half", "sector",
    "bonds", "defaulted", "upgrade", "same", "downgrade", "non_rated", "default_rate",
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[required_cols].copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["date", "sector"]).reset_index(drop=True)

print(df.shape)
df.head()


Using: data\SP_monthly_sector_and_all.csv
(6846, 13)


,date,year,month,quarter,half,sector,bonds,defaulted,upgrade,same,downgrade,non_rated,default_rate
0,1981-01-31,1981,1,1,1,ALL,1342,0,4,1290,45,3,0.0
1,1981-01-31,1981,1,1,1,Consumer,226,0,0,217,8,1,0.0
2,1981-01-31,1981,1,1,1,Energy,116,0,1,108,7,0,0.0
3,1981-01-31,1981,1,1,1,FI,87,0,0,86,1,0,0.0
4,1981-01-31,1981,1,1,1,Forest,68,0,0,63,5,0,0.0
